# Data Analysis of the filtered, encoded Bookings dataset

---

In [ ]:
import duckdb
import matplotlib.pyplot as plt
from dython.nominal import associations

from src.data_preprocessing.config import INPUT_FILE_BOOKINGS_SL_enc_f, HEATMAP_FILE_BOOKINGS

In [ ]:
con = duckdb.connect()

In [ ]:
df = con.execute(f"SELECT * FROM '{INPUT_FILE_BOOKINGS_SL_enc_f}'").fetchdf()
print(f"Shape: {df.shape}")
print("Summary:")
print(df.describe(include='all'))
print("Column types:")
print(df.dtypes)

In [ ]:
def plot_and_save_heatmap(df, output_path):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    plt.figure(figsize=(20, 18))
    associations(df[numeric_cols], nominal_columns=[],
                 mark_columns=True, figsize=(20, 18),
                 cmap='coolwarm', annot=True, fmt='.2f',
                 plot=True, title="Correlation Heatmap")
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

print("Generating correlation heatmap...")
plot_and_save_heatmap(INPUT_FILE_BOOKINGS_SL_enc_f, HEATMAP_FILE_BOOKINGS)
print(f"Heatmap saved to {HEATMAP_FILE_BOOKINGS}")

### Time Difference Analysis

In [ ]:
con.execute(f"""
CREATE OR REPLACE TEMP TABLE temp_bookings AS
SELECT *, EPOCH(book_stamp) - EPOCH(created_at) AS time_diff
FROM '{INPUT_FILE_BOOKINGS_SL_enc_f}';
""")

summary = con.execute("""
SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE time_diff = 0) AS zero_diff_count,
    MIN(time_diff) AS min_diff,
    MAX(time_diff) AS max_diff,
    AVG(time_diff) AS mean_diff,
    STDDEV_SAMP(time_diff) AS std_diff
FROM temp_bookings;
""").fetchdf()
print("\nTime Difference Summary:\n", summary)

### NULL percentages per column

In [ ]:
columns = con.execute("DESCRIBE temp_bookings").fetchdf()['column_name'].tolist()
total_rows = con.execute("SELECT COUNT(*) FROM temp_bookings").fetchone()[0]
null_results = []
for col in columns:
    null_count = con.execute(f"SELECT COUNT(*) - COUNT({col}) FROM temp_bookings").fetchone()[0]
    null_percentage = round(100.0 * null_count / total_rows, 2)
    null_results.append((col, null_percentage))
null_results.sort(key=lambda x: x[1], reverse=True)

print("\n=== Missing Values (% by column) ===")
for col, perc in null_results:
    print(f"{col}: {perc:.2f}%")

### Cardinality (# of unique values per column)

In [ ]:
print("\n=== Cardinality (Number of Unique Values) ===")
for col in columns:
    unique_count = con.execute(f"SELECT COUNT(DISTINCT {col}) FROM temp_bookings").fetchone()[0]
    print(f"{col}: {unique_count}")

### Top 10 frequent values for categorical/object columns

In [ ]:
categorical_cols = ['part_group', 'line_id', 'serial_number_id', 'book_state', 'station_id']
print("\n=== Top 10 Frequent Values Per Categorical Column ===")
for col in categorical_cols:
    print(f"\n--- {col} ---")
    freq = con.execute(f"""
        SELECT {col}, COUNT(*) AS freq
        FROM temp_bookings
        GROUP BY {col}
        ORDER BY freq DESC
        LIMIT 10;
    """).fetchdf()
    print(freq)

# === Schema ===
schema = con.execute("DESCRIBE temp_bookings").fetchdf()
print("\n=== Dataset Schema ===\n", schema)

In [ ]:
con.close()